# Knowledge Engineering: The Flaw of Pure Statistics vs. The Power of Logic

In this notebook, we will explore a critical flaw in modern Large Language Models (LLMs) known as **Hallucination**. Because LLMs at core predict answers based on statistical probability rather than hard facts, they can confidently invent dangerous medical advice or fake authoritative sources.

We will explore:
1. How LLMs fail under standard settings (**Default Baseline Hallucination**).
2. How tweaking parameters (`temperature=0.0`) still fails to fix the root issue (**Source Hallucination**).
3. How building a **Neo4j Knowledge Graph** with multiple medical domains—and querying it with a **Text2Cypher Retrieval Pipeline**—acts as a dynamic factual guardrail that retrieves *only* relevant facts while ignoring unrelated database noise.

## Part 1: Environment Setup
First, we need to set up our AI environment and graph database. We are using **Ollama** to run our local Llama model, and connecting to **Neo4j AuraDB** using `langchain-neo4j`.

*Note: Ensure your local Ollama application is running in the background and that your `neo4j_credentials.txt` is in the same directory with your `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD`.*

In [ ]:
# Install required dependencies in the current Jupyter kernel
# Uncomment the line below if you haven't installed these in your Anaconda environment yet
# %pip install -q ollama langchain-ollama neo4j langchain-neo4j pandas python-dotenv


import os
import pandas as pd
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

print("\nPackages loaded successfully.")

print("=== 1. Loading Neo4j AuraDB Credentials ===")

# Load the Neo4j credentials file
load_dotenv("neo4j_credentials.txt")

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

# Automatically patch strict SSL for local Windows environments
if URI and URI.startswith("neo4j+s://"):
    URI = URI.replace("neo4j+s://", "neo4j+ssc://")
    print("Auto-patched URI to use neo4j+ssc:// for local execution.")

try:
    # Initialize the graph connection
    graph = Neo4jGraph(url=URI, username=USER, password=PWD, enhanced_schema=True)
    graph.refresh_schema()
    print("\nSUCCESS! Connected to Neo4j AuraDB.")
    print("\n=== DETECTED GRAPH SCHEMA ===")
    print(graph.schema)
except Exception as e:
    print(f"\nCONNECTION FAILED! Please check your neo4j_credentials.txt file.\nError:\n{e}")

def run_cypher(query, params=None):
    return pd.DataFrame(graph.query(query, params=params or {}))

## Part 2: Act 1 - The Default Baseline Hallucination
Let's ask our AI a highly specific medical question regarding a severe drug interaction between **Warfarin** and **Celecoxib** in a patient with a specific genetic phenotype (**CYP2C9 poor metabolizer**).

Using **default settings**, the model guesses the most statistically common words. Watch how it confidently suggests dangerous alternative medications (like Ibuprofen, Naproxen, or Diclofenac) that actually cause severe internal bleeding when mixed with Warfarin. 

> **Execution Note:** Because LLMs rely on probabilistic sampling by default, the exact phrasing and specific dangerous drug recommendations will vary each time you run this cell. This inherent inconsistency demonstrates exactly why statistical guessing is unsafe for clinical logic.

In [ ]:
import ollama
from IPython.display import display, Markdown

MODEL_NAME = "llama3.2:3b" # You can also use "llama3.1"

prompt = (
    "A patient is taking Warfarin and has a CYP2C9 poor metabolizer phenotype. "
    "They need an anti-inflammatory for acute pain. Is Celecoxib safe to prescribe for this patient?"
)

print("=" * 60)
print("1. DEFAULT MODEL RESPONSE (Dangerous Guessing)")
print("=" * 60)

response_default = ollama.chat(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': prompt}]
)

display(Markdown(response_default['message']['content']))

**Clinical Note on the Hallucination:**
For non-medical students, mixing Warfarin (a blood thinner) with NSAIDs like Ibuprofen, Naproxen, or Diclofenac drastically increases the risk of severe, life-threatening internal bleeding. Furthermore, Acetaminophen (Paracetamol) is an analgesic/antipyretic, *not* an NSAID. The AI confidently prescribed a dangerous combination because it was statistically guessing common pain medications without understanding strict clinical contraindications.

## Part 3: Act 2 - The Deterministic Baseline
Can we fix this by just tweaking the model's parameters?

Let's set the `temperature` to `0.0` and use a fixed `seed`. This removes the AI's "creativity" and forces it to pick the most mathematically probable words.

While it might stop suggesting dangerous NSAIDs, watch how it either omits critical safe alternatives or **invents fake medical guidelines** (citing AASLD or ACG instead of the real CPIC guidelines) when pressed for authoritative reasoning.

In [ ]:
print("=" * 60)
print("2. DETERMINISTIC MODEL RESPONSE (Temperature = 0.0)")
print("=" * 60)

response_deterministic = ollama.chat(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': prompt}],
    options={'temperature': 0.0, 'seed': 42}
)

display(Markdown(response_deterministic['message']['content']))

**Note on Deterministic Limits:**
Lowering the temperature makes the model repeatable, but **predictability does not equal factual truth**. Without an external source of truth, a deterministic model still relies purely on compressed statistical weights.

## Part 4: Building a Knowledge Graph
To prevent the AI from guessing, we use **Knowledge Engineering** to explicitly map out clinical rules in Neo4j.

### Testing Selective Retrieval 
In a real hospital database, Warfarin and Celecoxib won't be the only medications stored. There will be thousands of other drugs, enzymes, and side effects. 

To prove that our **Text2Cypher** pipeline only retrieves the exact facts needed for our question (and doesn't just dump the whole database into the LLM), we will populate our graph with **three distinct medical domains**:
1. **Target Domain (Anticoagulants & Pain / CYP2C9)**: Warfarin, Celecoxib, NSAIDs, Acetaminophen, CPIC Guidelines.
2. **Distractor Domain 1 (Diabetes & Radiology)**: Metformin, Iodinated Contrast Dye, eGFR < 30 mL/min, Lactic Acidosis, ACR Guidelines.
3. **Distractor Domain 2 (Cholesterol & Antibiotics / CYP3A4)**: Simvastatin, Clarithromycin, CYP3A4 Enzyme, Rhabdomyolysis.

In [ ]:
# Clear previous demo data
run_cypher("MATCH (n) DETACH DELETE n")

seed_query = """
// ==========================================================================
// DOMAIN 1: TARGET ENTITIES (Warfarin, Celecoxib & CYP2C9 Pharmacogenomics)
// ==========================================================================
MERGE (cel:MedEntity {name: "Celecoxib", category: "NSAID (COX-2 Inhibitor)"})
MERGE (war:MedEntity {name: "Warfarin", category: "Anticoagulant"})
MERGE (cyp2c9:MedEntity {name: "CYP2C9 Enzyme", category: "Metabolic Enzyme"})
MERGE (bleed:MedEntity {name: "Severe Gastrointestinal Bleeding", category: "Adverse Event"})
MERGE (cel_war:MedEntity {name: "Celecoxib + Warfarin", category: "Drug Combination"})
MERGE (cpic:MedEntity {name: "Clinical Pharmacogenetics Implementation Consortium (CPIC)", category: "Organization"})
MERGE (pheno:MedEntity {name: "CYP2C9 Poor Metabolizer Phenotype", category: "Genetic Phenotype"})
MERGE (guide:MedEntity {name: "CPIC Guidelines", category: "Clinical Guideline"})
MERGE (nsaid:MedEntity {name: "NSAIDs (Ibuprofen, Naproxen)", category: "Drug Class"})
MERGE (ace:MedEntity {name: "Acetaminophen (Paracetamol)", category: "Analgesic"})
MERGE (war_pat:MedEntity {name: "Warfarin Patients", category: "Patient Group"})

// Target Domain Relationships
MERGE (cel)-[:METABOLIZED_BY]->(cyp2c9)
MERGE (war)-[:METABOLIZED_BY]->(cyp2c9)
MERGE (cel_war)-[:INCLUDES]->(cel)
MERGE (cel_war)-[:INCLUDES]->(war)
MERGE (cel_war)-[:CAUSES_HIGH_RISK_OF]->(bleed)
MERGE (pheno)-[:REDUCES_ACTIVITY_OF]->(cyp2c9)
MERGE (pheno)-[:INCREASES_TOXICITY_RISK_OF]->(cel)
MERGE (pheno)-[:INCREASES_BLEEDING_RISK_OF]->(war)
MERGE (guide)-[:PUBLISHED_BY]->(cpic)
MERGE (guide)-[:STRONGLY_RECOMMENDS_AVOIDING]->(cel_war)
MERGE (guide)-[:APPLIES_TO]->(pheno)
MERGE (nsaid)-[:INTERACTS_DANGEROUSLY_WITH]->(war)
MERGE (ace)-[:IS_PREFERRED_SAFE_ANALGESIC_FOR]->(war_pat)
MERGE (war_pat)-[:TAKES]->(war)

// ==========================================================================
// DOMAIN 2: DISTRACTOR ENTITIES (Metformin & Contrast Dye - Endocrinology)
// ==========================================================================
MERGE (met:MedEntity {name: "Metformin", category: "Antidiabetic"})
MERGE (contrast:MedEntity {name: "Iodinated Contrast Dye", category: "Radiocontrast Agent"})
MERGE (met_con:MedEntity {name: "Metformin + Iodinated Contrast Dye", category: "Drug Combination"})
MERGE (lactic:MedEntity {name: "Lactic Acidosis", category: "Adverse Event"})
MERGE (egfr:MedEntity {name: "Renal Impairment (eGFR < 30 mL/min)", category: "Clinical Condition"})
MERGE (acr:MedEntity {name: "ACR Contrast Manual Guidelines", category: "Clinical Guideline"})

// Distractor Domain 1 Relationships
MERGE (met_con)-[:INCLUDES]->(met)
MERGE (met_con)-[:INCLUDES]->(contrast)
MERGE (met_con)-[:CAUSES_HIGH_RISK_OF]->(lactic)
MERGE (egfr)-[:INCREASES_TOXICITY_RISK_OF]->(met)
MERGE (acr)-[:STRONGLY_RECOMMENDS_WITHHOLDING]->(met)

// ==========================================================================
// DOMAIN 3: DISTRACTOR ENTITIES (Simvastatin & Clarithromycin - Cardiology)
// ==========================================================================
MERGE (sim:MedEntity {name: "Simvastatin", category: "Statin"})
MERGE (cla:MedEntity {name: "Clarithromycin", category: "Macrolide Antibiotic"})
MERGE (cyp3a4:MedEntity {name: "CYP3A4 Enzyme", category: "Metabolic Enzyme"})
MERGE (rhabdo:MedEntity {name: "Rhabdomyolysis (Severe Muscle Breakdown)", category: "Adverse Event"})

// Distractor Domain 2 Relationships
MERGE (sim)-[:METABOLIZED_BY]->(cyp3a4)
MERGE (cla)-[:STRONGLY_INHIBITS]->(cyp3a4)
MERGE (cla)-[:INTERACTS_DANGEROUSLY_WITH]->(sim)
MERGE (sim)-[:CAUSES_HIGH_RISK_OF]->(rhabdo)
"""

graph.query(seed_query)
graph.refresh_schema()

# Let's inspect the total database size to see all the entities we just loaded
total_nodes = run_cypher("MATCH (n:MedEntity) RETURN count(n) AS Total_Nodes").iloc[0]['Total_Nodes']
total_rels = run_cypher("MATCH ()-[r]->() RETURN count(r) AS Total_Relationships").iloc[0]['Total_Relationships']

print(f"Neo4j Medical Knowledge Graph populated successfully!")
print(f"Total MedEntity Nodes in DB: {total_nodes} (Includes Target + Distractor Drugs)")
print(f"Total Clinical Relationships in DB: {total_rels}")

# Display all nodes in the database grouped by domain category
display(run_cypher("MATCH (n:MedEntity) RETURN n.name AS Entity_Name, n.category AS Category ORDER BY n.category"))

## Part 5: Act 3 - Dynamic GraphRAG via Text2Cypher (With Stage-by-Stage Inspection)

Teaching a smaller local LLM (like Llama 3.2 or 3.1) to write robust Cypher queries, we will apply three engineering best practices:

1. **Normalize Entities:** Use Cypher's `toLower()` function to make case-insensitive comparisons.
2. **Fuzzy Matching:** Use `CONTAINS` instead of strict equality `=` to capture multi-word concepts safely.
3. **Directed Edges (No Duplicates):** Enforce directed relationships `-[r]->` instead of undirected `-[]-` to prevent Neo4j from returning duplicate, reversed rows that waste the LLM's context window.

```text
[Stage 1: Input] 
   User Question + Neo4j Graph Schema (Node labels & Relationship types, NO data rows)
        │
        ▼
[Stage 2: Cypher Generation (cypher_llm)]
   LLM extracts entities from the User Question and writes a targeted Cypher Query
        │
        ▼
[Stage 3: Database Execution (Neo4j AuraDB)]
   Neo4j runs the Cypher query and returns ONLY matching triples (filtering out Metformin/Simvastatin)
        │
        ▼
[Stage 4: Grounded Answer Synthesis (qa_llm)]
   User Question + Retrieved Graph Triples -> Safe, Fact-Grounded Clinical Answer
```

In [ ]:
from langchain_neo4j import GraphCypherQAChain
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

# 1. Initialize our local deterministic LLM
llm = ChatOllama(model=MODEL_NAME, temperature=0.0)

# 2. Prompt for Stage 2 (Text-to-Cypher Translation)
# This forces the LLM to extract the actual drug/gene keywords from {question}.
cypher_prompt = PromptTemplate(
    template="""Task: Generate a Neo4j Cypher query to retrieve relationships for the medical entities mentioned in the User Question.

Graph Schema:
{schema}

Instructions:
1. Identify the specific drug names and gene/enzyme names in the User Question below.
2. Use ONLY single-word lowercase root keywords (for example, use 'cyp2c9' instead of 'cyp2c9 poor metabolizer phenotype').
3. Fill those keywords into this exact Cypher structure:
MATCH (n:MedEntity)-[r]->(m:MedEntity)
WHERE toLower(n.name) CONTAINS '<keyword1>' OR toLower(m.name) CONTAINS '<keyword1>'
   OR toLower(n.name) CONTAINS '<keyword2>' OR toLower(m.name) CONTAINS '<keyword2>'
   OR toLower(n.name) CONTAINS '<keyword3>' OR toLower(m.name) CONTAINS '<keyword3>'
RETURN DISTINCT n.name AS source, type(r) AS relationship, m.name AS target

4. Output ONLY the executable Cypher query. Do not include markdown code blocks or explanations.

User Question: {question}
Cypher Query:""",
    input_variables=["schema", "question"]
)

# 3. Prompt for Stage 4 (Final Answer Synthesis from Retrieved Context)

qa_prompt = PromptTemplate(
    template="""You are a clinical decision support assistant.
Answer the User Question using ONLY the Retrieved Knowledge Graph Facts provided below.

Rules:
- State clearly whether the medication combination is safe or unsafe.
- Explain the specific clinical and metabolic risks shown in the facts.
- Cite any official clinical guidelines present in the facts.
- State any preferred safe alternative medications if they are mentioned in the facts.
- Do NOT invent facts or mention medications not present in the Retrieved Knowledge Graph Facts.

Retrieved Knowledge Graph Facts:
{context}

User Question: {question}
Grounded Clinical Answer:""",
    input_variables=["context", "question"]
)

# 4. Build the Text2Cypher chain with `return_intermediate_steps=True`
cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm,
    qa_llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,
    qa_prompt=qa_prompt,
    verbose=False,
    return_intermediate_steps=True,
    allow_dangerous_requests=True
)

# 5. Run the pipeline on our Warfarin + Celecoxib + CYP2C9 prompt!
rag_response = cypher_chain.invoke({"query": prompt})
print("Pipeline execution complete! Run the cell below to inspect every stage.")

### Inspecting What Happens Inside Each Stage of the Pipeline
Let's unpack `rag_response` to see the exact data passed between the User, the LLM, and the Neo4j Graph Database at each step—and verify that:
1. **Stage 2** dynamically extracted `'warfarin'`, `'celecoxib'`, and `'cyp2c9'` from the user prompt.
2. **Stage 3** retrieved all 14 target relationships while filtering out **100%** of the distractor nodes (*Metformin*, *Simvastatin*, *Clarithromycin*).

In [ ]:
# Extract the intermediate steps returned by GraphCypherQAChain
generated_cypher = rag_response["intermediate_steps"][0]["query"]
retrieved_context = rag_response["intermediate_steps"][1]["context"]
final_answer = rag_response["result"]

print("=" * 80)
print("STAGE 1: INFORMATION PASSED TO THE CYPHER GENERATOR LLM")
print("=" * 80)
print("1A. User Question:")
print(f"    '{prompt}'\n")
print("1B. Graph Schema (Note: Only structure is passed, NOT the actual database rows!):")
print(graph.schema)

print("=" * 80)
print("STAGE 2: CYPHER QUERY GENERATED BY THE LLM (FROM ACTUAL USER QUESTION)")
print("=" * 80)
print(generated_cypher.strip())

print("\n" + "=" * 80)
print("STAGE 3: EXACT SUBGRAPH RETRIEVED FROM NEO4J (FORMATTED AS TABLE)")
print("=" * 80)
df_context = pd.DataFrame(retrieved_context)
display(df_context)

# Verify that distractor entities were NOT retrieved
distractors = ["Metformin", "Simvastatin", "Clarithromycin", "Lactic Acidosis", "Rhabdomyolysis"]
context_str = str(retrieved_context)
leaked_distractors = [d for d in distractors if d.lower() in context_str.lower()]

print(f"Total Relationships in Neo4j Database: {total_rels}")
print(f"Targeted Relationships Retrieved for Prompt: {len(df_context)}")
print(f"Distractor Entities Retrieved ({', '.join(distractors)}): {len(leaked_distractors)} (Noise Successfully Filtered!)")

print("\n" + "=" * 80)
print("STAGE 4: FINAL GROUNDED ANSWER (QA LLM SYNTHESIS)")
print("=" * 80)
display(Markdown(final_answer))

### Practice Exercise: Querying a Different Medical Domain With the Exact Same Chain

**Your Task**
1. Create a prompt for a patient with Renal Impairment who is taking Metformin and needs Iodinated Contrast Dye for a CT scan. Ask about the risks and what the guidelines recommend. 
2. Invoke your existing `cypher_chain` with this new prompt. 
3. Extract and print the generated Cypher query to verify it extracted the correct entities (Metformin, Iodinated Contrast Dye, etc.). 
4. Display the retrieved facts to ensure no unrelated drugs were fetched. 
5. Print the final grounded clinical answer.

In [ ]:
# Placeholder for code